# Churn

Logo churn and revenue churn from the Tidemill engine.

## How churn is measured

**Logo (customer) churn** — the fraction of *customers* who fully cancel:

```
logo_churn = customers_fully_churned / customers_active_at_period_start
```

A customer is "active at start" if they had positive MRR at the period
start (cumulative MRR movements before start > 0).  They are "fully
churned" if, by the period end, they have zero active subscriptions
remaining.

**Revenue churn** — the fraction of *MRR* lost to cancellations:

```
revenue_churn = |churned_MRR| / starting_MRR
```

`churned_MRR` sums the MRR of every subscription whose churn event
falls within `[start, end)`.

In [1]:
from tidemill import reports
from tidemill.reports.client import TidemillClient

reports.setup()

START, END = "2025-08-01", "2026-03-31"
CHURN_START = "2025-08-01"
tm = TidemillClient()

## 1. Customer churn sets

**C_start** — customers active at period start (positive MRR).
**C_churned** — subset of C_start who fully churned during the window.

In [2]:
detail = reports.churn.customer_detail(tm, CHURN_START, END)
reports.churn.style_c_start(detail)

customer,customer_name,starting_mrr
cus_ULdk3iPcoI9CLe,Active Starter #2,$20.00
cus_ULdkfi3Mg9S9q6,Active Starter #1,$20.00
cus_ULdkhc6fTXOggE,Active Starter Heavy #3,$20.00
cus_ULdl1VQsedZtx8,Late Churned Starter #13,$20.00
cus_ULdlStejDAhcFK,Active Monthly Pro #4,$79.00
cus_ULdlTvyjAKX3Tv,Churned Starter #8,$20.00
cus_ULdlZ6ctuNsyXQ,Churned Pro #11,$79.00
cus_ULdlctVigue6Pv,Downgraded Pro→Starter #10,$79.00
cus_ULdlfzB8AGoSVW,Active Annual Enterprise #7,$207.50
cus_ULdloHJhrIWeJC,Upgraded Starter→Pro #12,$20.00


In [3]:
reports.churn.style_c_churned(detail)

customer,customer_name,starting_mrr,churned_mrr
TOTAL,,$0.00,$0.00


## 2. Churn Rates

Logo and revenue churn from the Tidemill API.  These should match the
C_start / C_churned sets above:

- **Logo churn** = |C_churned| / |C_start|
- **Revenue churn** = sum(churned MRR) / sum(starting MRR)

In [9]:
reports.churn.style_snapshot(reports.churn.snapshot(tm, CHURN_START, END, detail=detail))

Metric,Numerator,Denominator,Rate
Logo churn,0 customers,19 customers,0.0%
Revenue churn,$0.00,$908.33,0.0%


### Revenue churn detail

Every customer from C_start with their starting MRR and churned MRR.
Customers who cancelled one subscription but kept another appear here
with churned MRR > 0 even though they are not in C_churned.

In [10]:
reports.churn.style_revenue_events(
    reports.churn.revenue_events(tm, CHURN_START, END, detail=detail)
)

customer,customer_name,starting_mrr,churned_mrr,fully_churned
cus_ULdk3iPcoI9CLe,Active Starter #2,$20.00,$0.00,False
cus_ULdkfi3Mg9S9q6,Active Starter #1,$20.00,$0.00,False
cus_ULdkhc6fTXOggE,Active Starter Heavy #3,$20.00,$0.00,False
cus_ULdl1VQsedZtx8,Late Churned Starter #13,$20.00,$0.00,False
cus_ULdlStejDAhcFK,Active Monthly Pro #4,$79.00,$0.00,False
cus_ULdlTvyjAKX3Tv,Churned Starter #8,$20.00,$0.00,False
cus_ULdlZ6ctuNsyXQ,Churned Pro #11,$79.00,$0.00,False
cus_ULdlctVigue6Pv,Downgraded Pro→Starter #10,$79.00,$0.00,False
cus_ULdlfzB8AGoSVW,Active Annual Enterprise #7,$207.50,$0.00,False
cus_ULdloHJhrIWeJC,Upgraded Starter→Pro #12,$20.00,$0.00,False


## 3. Monthly Churn Rate Timeline

Compute logo and revenue churn rates per month to visualize churn trends over time.

In [11]:
churn_df = reports.churn.timeline(tm, CHURN_START, END)
reports.churn.style_timeline(churn_df)

,logo_churn,revenue_churn
month,,
2025-08,0.0%,0.0%
2025-09,0.0%,0.0%
2025-10,0.0%,0.0%
2025-11,0.0%,0.0%
2025-12,0.0%,0.0%
2026-01,0.0%,0.0%
2026-02,0.0%,0.0%


In [7]:
reports.churn.plot_timeline(churn_df)

## 4. Monthly Churned MRR

Lost MRR per month from the MRR waterfall — shows the dollar impact of churn over time.

In [8]:
reports.churn.plot_monthly_lost_mrr(reports.churn.monthly_lost_mrr(tm, START, END))